In [4]:
import numpy as np                
import pandas as pd               
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from prophet import Prophet

In [5]:
df_contactos = pd.read_pickle('data/df_contactos_clean.pkl')
df_agentes = pd.read_pickle('data/df_agentes_clean.pkl')
df_calendario = pd.read_pickle('data/df_calendario_clean.pkl')
df_campañas = pd.read_pickle('data/df_campañas_clean.pkl')

In [6]:
print(df_contactos.columns.tolist())
print(df_agentes.columns.tolist())
print(df_calendario.columns.tolist())
print(df_campañas.columns.tolist())

['id_contacto', 'fecha', 'hora_inicio', 'canal', 'departamento', 'duracion_segundos', 'resuelto', 'SLA_cumplido', 'campania', 'anio_mes', 'anio', 'mes']
['fecha', 'franja_horaria', 'agentes_programados', 'agentes_presentes', 'ausencias', 'vacaciones', 'anio_mes', 'hora_inicio']
['fecha', 'anio', 'mes', 'trimestre', 'semana', 'dia_semana', 'fin_semana', 'festivo', 'campania', 'anio_mes']
['campania', 'inicio', 'fin', 'incremento_volumen', 'incremento_AHT', 'mes_num']


In [7]:
# Target
df_modelo = df_contactos.groupby(['fecha', 'hora_inicio', 'departamento']).size().reset_index(name='total_contactos')

# Features de calendario
df_modelo = df_modelo.merge(
    df_calendario[['fecha', 'dia_semana', 'mes', 'anio', 'trimestre', 'fin_semana', 'festivo', 'campania']],
    on='fecha',
    how='left'
)

# hora_inicio como categórica
df_modelo['hora_inicio'] = df_modelo['hora_inicio'].astype('category')

# Indicador booleano de campaña
df_modelo['es_campania'] = df_modelo['campania'] != 'No'

# Lag: mismo departamento y hora, día anterior
df_modelo = df_modelo.sort_values(['departamento', 'hora_inicio', 'fecha'])
df_modelo['volumen_dia_anterior'] = df_modelo.groupby(['departamento', 'hora_inicio'])['total_contactos'].shift(1)

In [8]:
df_modelo.head()

,fecha,hora_inicio,departamento,total_contactos,dia_semana,mes,anio,trimestre,fin_semana,festivo,campania,es_campania,volumen_dia_anterior
0,2024-01-01,8,ATC,11,Lunes,1,2024,1,False,True,NaN,True,NaN
45,2024-01-02,8,ATC,11,Martes,1,2024,1,False,False,NaN,True,11.0
90,2024-01-03,8,ATC,8,Miércoles,1,2024,1,False,False,NaN,True,11.0
135,2024-01-04,8,ATC,11,Jueves,1,2024,1,False,False,NaN,True,8.0
180,2024-01-05,8,ATC,9,Viernes,1,2024,1,False,False,NaN,True,11.0


In [9]:
if df_modelo['campania'].dtype.name == 'category' and 'Sin campaña' not in df_modelo['campania'].cat.categories:
    df_modelo['campania'] = df_modelo['campania'].cat.add_categories('Sin campaña')

df_modelo['campania'] = df_modelo['campania'].fillna('Sin campaña')
df_modelo['es_campania'] = df_modelo['campania'] != 'Sin campaña'

In [10]:
print(df_modelo['campania'].isna().sum())  # debería dar 0
print(df_modelo['campania'].unique())

0
['Sin campaña', 'Rebajas', 'Black Friday', 'Navidad']
Categories (4, str): ['Black Friday', 'Navidad', 'Rebajas', 'Sin campaña']


In [11]:
print(df_modelo['volumen_dia_anterior'].isna().sum())  # cuántas filas se van a perder
df_modelo = df_modelo.dropna(subset=['volumen_dia_anterior'])

45


In [12]:
df_modelo = df_modelo.sort_values(['departamento', 'hora_inicio', 'fecha'])
df_modelo['volumen_semana_anterior'] = df_modelo.groupby(['departamento', 'hora_inicio'])['total_contactos'].shift(7)

In [13]:
print(df_modelo[['volumen_dia_anterior', 'volumen_semana_anterior']].isna().sum())

df_modelo = df_modelo.dropna(subset=['volumen_dia_anterior', 'volumen_semana_anterior'])

volumen_dia_anterior         0
volumen_semana_anterior    315
dtype: int64


In [14]:
df_modelo_encoded = pd.get_dummies(
    df_modelo,
    columns=['departamento', 'dia_semana', 'hora_inicio', 'campania'],
    drop_first=True
)

In [15]:
df_modelo_encoded.head()

,fecha,total_contactos,mes,anio,trimestre,fin_semana,festivo,es_campania,volumen_dia_anterior,volumen_semana_anterior,...,hora_inicio_16,hora_inicio_17,hora_inicio_18,hora_inicio_19,hora_inicio_20,hora_inicio_21,hora_inicio_22,campania_Navidad,campania_Rebajas,campania_Sin campaña
360,2024-01-09,13,1,2024,1,False,False,True,14.0,11.0,...,False,False,False,False,False,False,False,False,True,False
405,2024-01-10,12,1,2024,1,False,False,True,13.0,8.0,...,False,False,False,False,False,False,False,False,True,False
450,2024-01-11,11,1,2024,1,False,False,True,12.0,11.0,...,False,False,False,False,False,False,False,False,True,False
495,2024-01-12,10,1,2024,1,False,False,True,11.0,9.0,...,False,False,False,False,False,False,False,False,True,False
540,2024-01-13,9,1,2024,1,True,False,True,10.0,8.0,...,False,False,False,False,False,False,False,False,True,False


In [16]:
print(df_modelo_encoded.isna().sum().sum())
print(df_modelo_encoded.shape)

0
(32535, 35)


In [17]:
X = df_modelo_encoded.drop(columns=['fecha', 'total_contactos'])
y = df_modelo_encoded['total_contactos']

In [18]:
fechas = df_modelo_encoded['fecha']

In [19]:
fecha_corte = df_modelo_encoded['fecha'].quantile(0.8, interpolation='nearest')
# o, más directo, eligiendo tú una fecha concreta y "redonda":
# fecha_corte = pd.Timestamp('2025-08-01')

train_mask = df_modelo_encoded['fecha'] < fecha_corte
test_mask = df_modelo_encoded['fecha'] >= fecha_corte

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f'Train: {X_train.shape[0]} filas, hasta {df_modelo_encoded["fecha"][train_mask].max()}')
print(f'Test: {X_test.shape[0]} filas, desde {df_modelo_encoded["fecha"][test_mask].min()}')

Train: 26010 filas, hasta 2025-08-08 00:00:00
Test: 6525 filas, desde 2025-08-09 00:00:00


In [21]:
modelo_lr = LinearRegression()
modelo_rf = RandomForestRegressor(random_state=42)
modelo_xgb = XGBRegressor(random_state=42)